<a href="https://colab.research.google.com/github/nauzley/Clinical-Note-Evaluation/blob/main/Clinical_Note_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#-------------------------------------------------------------------------------
#                        OVERALL EVALUATION APPROACH
#-------------------------------------------------------------------------------
#                  ┌─────────────────────────────────────────┐
#                  │       DOCTOR - PATIENT DIALOGUE         |
#                  │            ACI-BENCH Dataset            |
#                  └────────────────────┬────────────────────┘
#                                       │
#                                       ▼
#                  ┌─────────────────────────────────────────┐
#                  │    AMBIENT AI NOTE GENERATION MODEL     |
#                  │            ACI-BENCH Dataset            |
#                  └────────────────────┬────────────────────┘
#                                       │
#                                       ▼
# ┌───────────────────────────────────────────────────────────────────────────┐
# │    PHASE 1: PRE-VETTING EVALUATION (AI  Note vs. Original Transcript      |
# |       (Eval will be performed using 2 APIs: OpenAI, Anthropic )           |
# ├───────────────────────────────────────────────────────────────────────────┤
# │  • Clinical Accuracy & Hallucination Audit (Omissions vs. Fabrications)   │
# │  • HCC / Risk Adjustment Capture Rate                                     │
# │  • Safety & Clinical Contradiction Flags                                  │
# └─────────────────────────────────────┬─────────────────────────────────────┘
#                                       │
#                                       ▼
#                  ┌─────────────────────────────────────────┐
#                  │ CLINICIAN REVIEWS, EDITS & SIGNS IN EHR |
#                  └────────────────────┬────────────────────┘
#                                       │
#                                       ▼
# ┌───────────────────────────────────────────────────────────────────────────┐
# │ PHASE 2: POST-VETTING EVAL (Pre-Signed AI Note vs. Final Revised/Signed)  │
# ├───────────────────────────────────────────────────────────────────────────┤
# │  • Normalized Levenshtein Edit Distance (Clinician Editing Burden)        │
# │  • Section-Specific Insertion / Deletion Ratios                           │
# │  • Token Retention Rate                                                   │
# └───────────────────────────────────────────────────────────────────────────┘

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import Levenshtein
import time
import random

from google.colab import userdata
from anthropic import Anthropic
import google.generativeai as genai # Reverted import for Gemini
from openai import OpenAI, RateLimitError, APIError # Import specific OpenAI error types
from anthropic import APIStatusError, OverloadedError # Only import specific Anthropic error types needed
from anthropic.types import TextBlock # Added TextBlock import for handling Anthropic responses
from google.api_core.exceptions import GoogleAPIError # Import GoogleAPIError for Gemini

# Map Colab secrets to system environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get('AnthropicClinDocEvalAPIKey')
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAIClinDocEvalAPIKey')
os.environ["GEMINI_API_KEY"] = userdata.get('Gemini_Clin_Doc_Eval_APIKey')

# Configure Gemini API key
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

# Initialize the clients
anthropic_client = Anthropic()
openai_client = OpenAI()
gemini_model = genai.GenerativeModel('gemini-3.6-flash')

# Test Anthropic with retry logic
max_retries_anthropic = 5
base_delay_anthropic = 1 # seconds
anthropic_response = None

for i in range(max_retries_anthropic):
    try:
        anthropic_response = anthropic_client.messages.create(
            model="claude-sonnet-5", # Corrected model name based on error suggestion
            max_tokens=1000,
            messages=[{"role": "user", "content": "Say hello!"}]
        )
        print("Anthropic says:", anthropic_response.content[0].text)
        break # If successful, break the loop
    except OverloadedError as e:
        if i < max_retries_anthropic - 1:
            delay = base_delay_anthropic * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"Anthropic API overloaded. Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering OverloadedError: {e}")
            # If you want to raise an error after max retries:
            # raise # Re-raise the exception if max retries are exceeded
            print("Skipping Anthropic API call due to persistent overload.")
            break # Exit loop if Anthropic persistently overloaded
    except APIStatusError as e:
        print(f"Anthropic API error: {e}")
        raise # Re-raise other API errors immediately

# Test OpenAI with retry logic
max_retries_openai = 5
base_delay_openai = 1 # seconds
openai_response = None

for i in range(max_retries_openai):
    try:
        openai_response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Say hello!"}]
        )
        print("OpenAI says:", openai_response.choices[0].message.content)
        break # If successful, break the loop
    except (RateLimitError, APIError) as e:
        if i < max_retries_openai - 1:
            delay = base_delay_openai * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"OpenAI API error ({type(e).__name__}). Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering OpenAI API error: {e}")
            raise # Re-raise the exception if max retries are exceeded

# Test Gemini with retry logic
max_retries_gemini = 5
base_delay_gemini = 1 # seconds
gemini_response = None

for i in range(max_retries_gemini):
    try:
        gemini_response = gemini_model.generate_content("Say hello!")
        print("Gemini says:", gemini_response.text)
        break # If successful, break the loop
    except GoogleAPIError as e: # Changed genai.APIError to GoogleAPIError
        if i < max_retries_gemini - 1:
            delay = base_delay_gemini * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"Gemini API error ({type(e).__name__}). Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering Gemini API error: {e}")
            raise # Re-raise the exception if max retries are exceeded

Anthropic says: Hello! 👋 How are you doing today? I'm here and ready to help with whatever you need—questions, brainstorming, writing, coding, or just a conversation. What's on your mind?
OpenAI says: Hello! How can I assist you today?
Gemini says: Hello! How can I help you today?


In [ ]:
#-----------------------------
  #SET UP ACI-BENCH DATASET
#-----------------------------

# Load the dataset directly from GitHub
url = "https://raw.githubusercontent.com/microsoft/clinical_visit_note_summarization_corpus/refs/heads/main/data/aci-bench/challenge_data/train.csv"
df = pd.read_csv(url)

# View basic dataset information
print(f"Total notes available: {len(df)}")
print("Columns in dataset:", df.columns.tolist())
print("Transcript source types:", df['dataset'].value_counts())

#Take a small sample (3 notes) to reduce computation costs
sample_df = df.head(3).copy()
print(f"Successfully loaded {len(sample_df)} encounter records.")

# Display the first transcript snippet
print("\n--- SAMPLE TRANSCRIPT SNIPPET (Note #1) ---")
print(sample_df['note'].iloc[0][:300] + "...")


Total notes available: 67
Columns in dataset: ['dataset', 'encounter_id', 'dialogue', 'note']
Transcript source types: dataset
aci           35
virtassist    20
virtscribe    12
Name: count, dtype: int64
Successfully loaded 3 encounter records.

--- SAMPLE TRANSCRIPT SNIPPET (Note #1) ---
CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who presents for her annual exam. It has been a year since I last saw the patient.

The patient has bee...


In [ ]:
#----------------------------------------------------------------------------
#PHASE 1: PRE-VETTING EVAL(AI Generated Note vs. Original Transcript)
#----------------------------------------------------------------------------

# Note: ACI-Bench findings demonstrated that LLMs could resolve transcription errors related to source (virtual assistant vs ASR vs virtual scribe).
# Hence transcript source was not considered as a variable in the evaluation of AI generated note quality.

# Universal Clinical Audit Prompt Template
AUDIT_PROMPT_TEMPLATE = """You are an expert Clinical Documentation Integrity (CDI) Auditor and Compliance Specialist.
Audit the following AI-generated draft note against the doctor-patient transcript.

### TRANSCRIPT:
{transcript}

### AI DRAFT NOTE:
{ai_note}

Evaluate on 3 specific compliance domains:
1. Accuracy (0-100): Subtract points for fabrications or critical clinical omissions.
2. HCC Capture (0-100): Check if chronic conditions have sufficient documentation specificity (MEAT criteria).
3. Safety (0-100): Check for internal medical contradictions.

STRICT REQUIREMENT: Reply ONLY in valid JSON matching this schema:
{{
  "accuracy_score": <int 0-100>,
  "hcc_capture_score": <int 0-100>,
  "safety_score": <int 0-100>,
  "key_findings": "<short summary of errors or strengths>"
}}"""

def evaluate_with_openai(transcript, ai_note):
    """Judge 1: OpenAI gpt-4o-mini"""
    prompt = AUDIT_PROMPT_TEMPLATE.format(transcript=transcript, ai_note=ai_note)

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)

def evaluate_with_anthropic(transcript, ai_note):
    """Judge 2: Anthropic claude-sonnet-5"""
    prompt = AUDIT_PROMPT_TEMPLATE.format(transcript=transcript, ai_note=ai_note)

    max_retries = 3
    for i in range(max_retries):
        try:
            response = anthropic_client.messages.create(
                model="claude-sonnet-5",  # Updated model name
                max_tokens=2000, # Increased max_tokens to allow for full response including thinking blocks
                messages=[{"role": "user", "content": prompt}]
            )

            raw_text = ""
            # Iterate through the content blocks to find the first TextBlock
            for block in response.content:
                # Check if the block is an instance of TextBlock
                if isinstance(block, TextBlock):
                    raw_text = block.text
                    break # Once a TextBlock is found, use its text and exit the loop

            # If no TextBlock was found in the response, raise an error to trigger retry or final error
            if not raw_text:
                raise ValueError(f"Anthropic response did not contain a TextBlock. Full content: {response.content}")

            clean_json = raw_text.replace("```json", "").replace("```", "").strip()
            return json.loads(clean_json)
        except ValueError as e:
            if i < max_retries - 1:
                print(f"Anthropic TextBlock not found error: {e}. Retrying...")
                time.sleep(2 ** i) # Exponential backoff
            else:
                raise # Re-raise if max retries reached
        except Exception as e: # Catch other potential API errors during call
            print(f"An unexpected error occurred during Anthropic API call: {e}. Retrying...")
            time.sleep(2 ** i)
    return {} # Should not be reached if retries are exhaustive, but good for safety


In [ ]:
transcript_to_evaluate = sample_df['dialogue'].iloc[0]
ai_note_to_evaluate = sample_df['note'].iloc[0]

# Evaluate with OpenAI
openai_evaluation_result = evaluate_with_openai(transcript_to_evaluate, ai_note_to_evaluate)
display(openai_evaluation_result)

# Evaluate with Anthropic
anthropic_evaluation_result = evaluate_with_anthropic(transcript_to_evaluate, ai_note_to_evaluate)
display(anthropic_evaluation_result)

{'accuracy_score': 90,
 'hcc_capture_score': 85,
 'safety_score': 95,
 'key_findings': "The note accurately reflects the patient's history and current status, but it omits specific details about the patient's medication adherence and the impact of stress on her hypertension. HCC capture is strong but could benefit from more detail on the management of her chronic conditions. No internal contradictions were found."}

{'accuracy_score': 85,
 'hcc_capture_score': 90,
 'safety_score': 95,
 'key_findings': "Fabrication: Patient surname 'Collins' appears in HPI and A&P but is never stated in the transcript (only 'Martha' is used) - this is an unsupported addition to patient identification. Minor omission: The doctor's explicit patient education regarding combining pill administration times (patient's question about taking all medications together) is not captured in the note despite being clinically relevant counseling. Depression section lacks explicit 'Medical Treatment' plan documentation (i.e., decision not to start medication), though this is implied by omission. HCC capture is strong overall: CHF is well-documented with EF 45%, systolic murmur, 1+ pitting edema, and treatment escalation (lisinopril increase, new Lasix) - supports HFrEF specificity. Hypertension has clear MEAT (monitoring via home BP logs, assessment of stress-related elevation, treatment with lisinopril increase). Depression has m

In [ ]:
all_eval_results = []

print("\n--- Evaluating first 3 sample notes ---")
for i in range(len(sample_df.head(3))):
    transcript_to_evaluate = sample_df['dialogue'].iloc[i]
    ai_note_to_evaluate = sample_df['note'].iloc[i]

    print(f"\n--- Sample Note {i+1} ---")

    # Evaluate with OpenAI
    print("OpenAI Evaluation:")
    openai_evaluation_result = evaluate_with_openai(transcript_to_evaluate, ai_note_to_evaluate)
    openai_evaluation_result['model'] = 'OpenAI'
    openai_evaluation_result['sample_id'] = i + 1
    all_eval_results.append(openai_evaluation_result)
    display(openai_evaluation_result)

    # Evaluate with Anthropic
    print("Anthropic Evaluation:")
    anthropic_evaluation_result = evaluate_with_anthropic(transcript_to_evaluate, ai_note_to_evaluate)
    anthropic_evaluation_result['model'] = 'Anthropic'
    anthropic_evaluation_result['sample_id'] = i + 1
    all_eval_results.append(anthropic_evaluation_result)
    display(anthropic_evaluation_result)

print("\n--- Aggregating results ---")
aggregated_df = pd.DataFrame(all_eval_results)
display(aggregated_df)


--- Evaluating first 3 sample notes ---

--- Sample Note 1 ---
OpenAI Evaluation:


{'accuracy_score': 90,
 'hcc_capture_score': 85,
 'safety_score': 95,
 'key_findings': "The note accurately reflects the patient's history and current status, but it omits specific details about the patient's medication adherence and the impact of stress on her hypertension. HCC capture is good but could benefit from more specificity regarding the management of chronic conditions. No internal contradictions were found.",
 'model': 'OpenAI',
 'sample_id': 1}

Anthropic Evaluation:


{'accuracy_score': 88,
 'hcc_capture_score': 85,
 'safety_score': 95,
 'key_findings': "Fabricated patient surname 'Collins' not present in transcript. Lung exam was mentioned as performed by the physician but findings were omitted from the Physical Exam section (only cardiovascular findings documented). Patient's question about consolidating medication timing was omitted from the plan despite being clinically relevant to her hypertension non-adherence issue. CHF, depression, and hypertension are documented with reasonable MEAT criteria (monitoring via BP/echo, evaluation via exam findings, treatment via medication changes), but CHF could be further specified as HFmrEF (EF 45%) for improved HCC specificity rather than just 'reduced ejection fraction.' No major internal contradictions were found; medication changes (lisinopril 40mg, Lasix 20mg) are consistent between HPI, exam findings, and assessment/plan.",
 'model': 'Anthropic',
 'sample_id': 1}


--- Sample Note 2 ---
OpenAI Evaluation:


...

{'accuracy_score': 90,
 'hcc_capture_score': 85,
 'safety_score': 95,
 'key_findings': "The note accurately reflects the patient's history and current issues, but it omits the patient's report of abdominal pain and the specific nature of the dizziness. HCC capture is good but could improve with more detail on the patient's chronic conditions. No internal contradictions were found.",
 'model': 'OpenAI',
 'sample_id': 3}

Anthropic Evaluation:


{'accuracy_score': 78,
 'hcc_capture_score': 70,
 'safety_score': 82,
 'key_findings': "Fabrication: HPI/ROS state patient 'endorses nausea and vomiting' with exertion, but transcript shows patient only described dizziness/lightheadedness and increased abdominal pain with exertion—no nausea or vomiting was ever mentioned. This is a clinically significant misrepresentation of symptoms relevant to obstructive uropathy workup. ROS also fabricates 'Denies headaches' under Neurological, which was not explicitly stated (patient only confirmed no issues with migraines, not a specific headache denial). Physical exam misclassifies CVA tenderness under 'Gastrointestinal' rather than Genitourinary/Renal, a categorization inconsistency. Otherwise, HPI, assessment, and plan are largely accurate and well-aligned with the transcript (kidney stone workup, CT order, Ultram, Imitrex continuation, Protonix refill). Chronic conditions (kidney stones, migraines, GERD) are documented with reasonable monitor


--- Aggregating results ---


,accuracy_score,hcc_capture_score,safety_score,key_findings,model,sample_id
0,90,85,95,The note accurately reflects the patient's his...,OpenAI,1
1,88,85,95,Fabricated patient surname 'Collins' not prese...,Anthropic,1
2,85,90,95,The note accurately reflects the patient's his...,OpenAI,2
3,78,85,80,1) Internal contradiction: HPI states pain 'do...,Anthropic,2
4,90,85,95,The note accurately reflects the patient's his...,OpenAI,3
5,78,70,82,Fabrication: HPI/ROS state patient 'endorses n...,Anthropic,3


In [ ]:
#Compare and contrast major similarities and differences between the evaluations of OpenAI and Anthropic

In [ ]:
#-------------------------------------------------------
# GENERATE PHYSICIAN FINAL REVIEWED/EDITED/SIGNED NOTES
#-------------------------------------------------------
#Note: In the real-world, we would have access to notes that were initially written by AI and underwent finalization (review, editting where appropriate, and signature) by an experienced clinician.
#In this case, we do not have such notes. Hence, we are calling a third API (Gemini) to act as a "master clinician" to finalize the notes and provide a gold standard for comparison in phase 2 of our evaluation.

def generate_master_clinician_gold_note(transcript, original_ai_note):
    """
    Uses Gemini as a Master Clinician & Compliance Auditor (15+ yrs experience)
    to revise, complete, and sign the original AI draft note into a Gold-Standard Note.
    """
    system_prompt = """You are a Master Clinician and Medical Billing/Compliance Auditor with over 15 years of clinical and documentation expertise.
Your task is to take an original AI-generated draft note and revise, edit, and perfect it against the encounter transcript to create the GOLD-STANDARD signed clinical note.
Do not make revisions if the note has high fidelity/accuracy with regards to clinical reasoning and comparison to the original transcipt, appropriate structure, and billing/risk adjustment components.

### REVISION REQUIREMENTS:
1. **Absolute Transcript Fidelity:** Remove all fabrications, unverified claims, or hallucinations. Keep only facts supported by the transcript.
2. **Clinical Rigor & HCC Billing Capture:** If missing in the AI draft, ensure the Assessment & Plan includes high-level clinical reasoning and full specificity (MEAT criteria) for all chronic conditions discussed.
3. **Structure:** Notes should be structured as SOAP notes.
4. **Attestation:** End the note with an official physician sign-off line. Do not include this addition in your comparison of note length/token changes.

Return ONLY the finalized, gold-standard clinical note."""

    user_prompt = f"""### ENCOUNTER TRANSCRIPT:
{transcript}

### ORIGINAL AI DRAFT NOTE TO REVISE:
{original_ai_note}

Please generate the finalized, revised Gold-Standard Note now:"""

    # Call Gemini API
    response = gemini_model.generate_content(
        contents=f"{system_prompt}\n\n{user_prompt}"
    )
    return response.text.strip()

In [ ]:
gold_notes_data = []

print("\n--- Generating Master Clinician Gold Standard Notes ---")
for i in range(len(sample_df.head(3))):
    transcript = sample_df['dialogue'].iloc[i]
    original_ai_note = sample_df['note'].iloc[i]

    print(f"\nGenerating Gold Note for Sample {i+1}...")
    gold_note = generate_master_clinician_gold_note(transcript, original_ai_note)

    gold_notes_data.append({
        'sample_id': i + 1,
        'transcript': transcript,
        'original_ai_note': original_ai_note,
        'gold_note': gold_note
    })

master_clinician_df = pd.DataFrame(gold_notes_data)
display(master_clinician_df[['sample_id', 'transcript', 'original_ai_note', 'gold_note']])


--- Generating Master Clinician Gold Standard Notes ---

Generating Gold Note for Sample 1...

Generating Gold Note for Sample 2...

Generating Gold Note for Sample 3...


,sample_id,transcript,original_ai_note,gold_note
0,1,"[doctor] hi , martha . how are you ?\n[patient...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...,**PATIENT:** Martha \n**AGE:** 50-year-old fe...
1,2,"[doctor] hi , andrew , how are you ?\n[patient...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...,**SUBJECTIVE**\n\n**Chief Complaint:**\nJoint ...
2,3,"[doctor] hi , john . how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...,**PATIENT DEMOGRAPHICS** \n**Name:** John Per...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
#---------------------------------------------------------------------------
#PHASE 2: POST-VETTING EVAL (Pre-Signed AI Note vs. Final Revised/Signed)
#---------------------------------------------------------------------------

def compute_post_vetting_edits(original_ai_note, gold_note):
    """
    Measures how much the synthetic attending physician (Dr. Gemini) had to edit
    the raw AI draft note before signing it.
    """
    raw_distance = Levenshtein.distance(original_ai_note, gold_note)
    max_len = max(len(original_ai_note), len(gold_note))
    normalized_edit_dist = raw_distance / max_len if max_len > 0 else 0.0

    draft_words = original_ai_note.split()
    signed_words = gold_note.split()
    word_distance = Levenshtein.distance(draft_words, signed_words)
    max_words = max(len(draft_words), len(signed_words))
    word_edit_ratio = word_distance / max_words if max_words > 0 else 0.0

    return {
        "character_edit_distance": raw_distance,
        "normalized_edit_distance": round(normalized_edit_dist, 4),
        "word_edit_ratio": round(word_edit_ratio, 4)
    }

In [ ]:
edit_results = []
for index, row in master_clinician_df.iterrows():
    original_ai_note = row['original_ai_note']
    gold_note = row['gold_note']
    edits = compute_post_vetting_edits(original_ai_note, gold_note)
    edit_results.append(edits)

edits_df = pd.DataFrame(edit_results)
master_clinician_df = pd.concat([master_clinician_df, edits_df], axis=1)
display(master_clinician_df[['sample_id', 'character_edit_distance', 'normalized_edit_distance', 'word_edit_ratio']])

,sample_id,character_edit_distance,normalized_edit_distance,word_edit_ratio
0,1,3808,0.7030,0.9337
1,2,2545,0.6769,0.8800
2,3,3028,0.6708,0.8954
